# F5-TTS — O'zbek ayol ovoz (FeruzaSpeech) fine-tune

**Maqsad:** FeruzaSpeech (toza professional ayol, 60h) datasida F5-TTS modelni o'rgatib,
tabiiy + iliq + tiniq o'zbek ayol ovozini olish.

**Foydalanish:** Colab'da **Runtime → Change runtime type → T4 GPU** ni tanlang, keyin
kataklarni ketма-ket ishga tushiring (▶). Savol/xato bo'lsa — natijani nusxalab menga yuboring.

F5-TTS lotin o'zbek matnда to'g'ridan-to'g'ri o'rganadi (transliteratsiya kerak emas).

## 1. GPU tekshirish

In [ ]:
!nvidia-smi

## 2. F5-TTS o'rnatish (~3-5 daqiqa)

In [ ]:
!pip install -q f5-tts soundfile librosa
print('F5-TTS o\'rnatildi')

## 3. Google Drive ulash (checkpoint saqlash uchun)
Training uzoq — Colab uzilsa, checkpoint Drive'da saqlanib qoladi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/f5_uzbek', exist_ok=True)
print('Drive ulandi: /content/drive/MyDrive/f5_uzbek')

## 4. HuggingFace login (FeruzaSpeech uchun)
Tokeningizni qo'ying (https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import login
login('hf_PASTE_YOUR_TOKEN_HERE')   # <-- tokeningizni qo'ying
print('HF login OK')

## 5. FeruzaSpeech yuklash + tayyorlash (24kHz, metadata.csv)
5-15s kliplar tanlanadi (F5 uchun ideal). Lotin matn ishlatiladi.

In [ ]:
import os, csv, soundfile as sf, librosa, numpy as np
from pathlib import Path
from huggingface_hub import snapshot_download
os.environ['HF_HUB_DISABLE_XET'] = '1'

RAW = '/content/feruza_raw'
snapshot_download('k2speech/FeruzaSpeech', repo_type='dataset', local_dir=RAW,
                  allow_patterns=['*.tsv', 'train/**'])
print('Yuklab olindi')

# metadata.csv (audio_file|text) + wavs/ (24kHz), 5-15s
DS = '/content/feruza_ds'; WAVS = f'{DS}/wavs'; os.makedirs(WAVS, exist_ok=True)
rows = list(csv.reader(open(f'{RAW}/train.tsv', encoding='utf-8'), delimiter='\t'))
h = rows[0]; ai=h.index('audio'); li=h.index('text_latin'); di=h.index('duration')
meta = []
for r in rows[1:]:
    if len(r) <= max(ai,li,di): continue
    try: dur = float(r[di])
    except: continue
    if dur < 4.0 or dur > 15.0: continue
    src = f'{RAW}/{r[ai]}'
    if not os.path.exists(src): continue
    name = r[ai].replace('/', '_')
    w, sr = sf.read(src)
    if w.ndim > 1: w = w.mean(1)
    if sr != 24000: w = librosa.resample(w.astype('float32'), orig_sr=sr, target_sr=24000)
    sf.write(f'{WAVS}/{name}', w, 24000, subtype='PCM_16')
    meta.append(f'wavs/{name}|{r[li].strip()}')
with open(f'{DS}/metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(meta))
print(f'{len(meta)} ta klip tayyor (24kHz). Namuna:'); print(meta[0])

## 6. Fine-tune interfeysini ochish (Gradio GUI)
Quyidagi katak F5-TTS fine-tune GUI'sini ochadi (public havola chiqadi).
GUI'da: **Tab 1** datasetni tayyorlash (`/content/feruza_ds` ni ko'rsating, vocab extend = ha),
**Tab 2** training (epochs ~ 50-100, batch GPU'ga qarab), checkpoint'ni Drive'ga yo'naltiring.

> Eslatma: GUI murakkab tuyulsa — menga screenshot/natija yuboring, qadam-baqadam aytaman.

In [ ]:
!f5-tts_finetune-gradio --share

## 7. Training tugagach
Checkpoint (`.pt` / `.safetensors`) Drive'da `f5_uzbek` papkasida bo'ladi.
Uni yuklab oling yoki menga yo'lini ayting — men sizning RTX 3060 server'ingizga
inference engine (`f5_engine.py`) yozib ulayman.